# Comparativa de Modelos
Evaluación de diferentes algoritmos (Random Forest, Gradient Boosting, SVM) con validación cruzada estratificada.

In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

## 1. Carga y Preparación de Datos

In [6]:
print("--- CARGANDO Y PREPARANDO DATOS ---")

df_gen = pd.read_csv('../../Datos/df_general.csv')

# Crear variable objetivo
df_gen['y'] = df_gen['Key'].map({
    'bonafide': 0,
    'spoof': 1
})

# Separar X e y
drop_columns = ['Key', 'file_name', 'User_ID', 'Spoofing_ID', 'y']

X = df_gen.drop(columns=drop_columns, errors='ignore')
y = df_gen['y']

print(f"Dataset cargado: {X.shape[0]} muestras y {X.shape[1]} variables\n")

--- CARGANDO Y PREPARANDO DATOS ---
Dataset cargado: 94632 muestras y 24 variables



## 2. Configuración de Cross Validation

In [7]:
# StratifiedKFold mantiene el balance de clases en cada fold
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

spoofing_ids = df_gen['Spoofing_ID']

# Métricas a evaluar
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score),
    'recall': make_scorer(recall_score),
    'f1': make_scorer(f1_score)
}

## 3. Definición de Modelos

In [ ]:
models = {
    # ---------------- RANDOM FOREST ----------------
    'Random Forest (50 árboles)': RandomForestClassifier(
        n_estimators=50,
        random_state=42,
        n_jobs=-1
    ),
    'Random Forest (100 árboles)': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
    'Random Forest (150 árboles)': RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        n_jobs=-1
    ),
    'Random Forest (200 árboles)': RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    'Random Forest (250 árboles / max_depth=5)': RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        max_depth=5,
        n_jobs=-1
    ),
        'Random Forest (250 árboles / max_depth=10)': RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        max_depth=10,
        n_jobs=-1
    ),
    'Random Forest (250 árboles / max_depth=20)': RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        max_depth=20,
        n_jobs=-1
    ),
        'Random Forest (250 árboles / max_depth=30)': RandomForestClassifier(
        n_estimators=250,
        random_state=42,
        max_depth=30,
        n_jobs=-1
    ),
    'Random Forest (300 árboles)': RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    # ---------------- GRADIENT BOOSTING ----------------
    'Gradient Boosting (50 árboles)': GradientBoostingClassifier(
        n_estimators=50,
        random_state=42
    ),
    'Gradient Boosting (100 árboles)': GradientBoostingClassifier(
        n_estimators=100,
        random_state=42
    ),
    # ---------------- SVM ----------------
    'SVM (linear kernel)': SVC(
        kernel='linear',
        random_state=42
    ),
    'SVM (rbf kernel)': SVC(
        kernel='rbf',
        random_state=42
    )
}

## 4. Evaluación con Cross Validation

In [16]:
print("==========================================")
print("EVALUACIÓN CON CROSS VALIDATION")
print("==========================================\n")

results = []

for model_name, model in models.items():
    print(f"--- Evaluando: {model_name} ---")

    # Pipeline:
    # 1. Escalado
    # 2. Undersampling SOLO en train de cada fold
    # 3. Modelo
    pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        ('undersampling', RandomUnderSampler(random_state=42)),
        ('model', model)
    ])

    # Cross Validation
    cv_results = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        groups=spoofing_ids,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    # Promedios
    accuracy_mean = np.mean(cv_results['test_accuracy'])
    precision_mean = np.mean(cv_results['test_precision'])
    recall_mean = np.mean(cv_results['test_recall'])
    f1_mean = np.mean(cv_results['test_f1'])

    # Desviación estándar
    accuracy_std = np.std(cv_results['test_accuracy'])
    f1_std = np.std(cv_results['test_f1'])

    results.append({
        'Modelo': model_name,
        'Accuracy': accuracy_mean,
        'Precision': precision_mean,
        'Recall': recall_mean,
        'F1-Score': f1_mean,
        'Std Accuracy': accuracy_std,
        'Std F1': f1_std
    })

    print(f"Accuracy : {accuracy_mean:.4f} (+/- {accuracy_std:.4f})")
    print(f"Precision: {precision_mean:.4f}")
    print(f"Recall   : {recall_mean:.4f}")
    print(f"F1-Score : {f1_mean:.4f}")
    print()

EVALUACIÓN CON CROSS VALIDATION

--- Evaluando: Random Forest (50 árboles) ---


c:\Users\marce\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:885: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


Accuracy : 0.8369 (+/- 0.0025)
Precision: 0.9897
Recall   : 0.8306
F1-Score : 0.9032

--- Evaluando: Random Forest (100 árboles) ---


c:\Users\marce\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:885: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


Accuracy : 0.8400 (+/- 0.0034)
Precision: 0.9899
Recall   : 0.8339
F1-Score : 0.9052

--- Evaluando: Random Forest (150 árboles) ---


...

Accuracy : 0.8168 (+/- 0.0026)
Precision: 0.9894
Recall   : 0.8087
F1-Score : 0.8900

--- Evaluando: SVM (linear kernel) ---


c:\Users\marce\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:885: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


Accuracy : 0.7624 (+/- 0.0026)
Precision: 0.9865
Recall   : 0.7508
F1-Score : 0.8527

--- Evaluando: SVM (rbf kernel) ---


c:\Users\marce\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:885: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


Accuracy : 0.8224 (+/- 0.0026)
Precision: 0.9932
Recall   : 0.8117
F1-Score : 0.8933



## 5. Tabla Final Comparativa

In [17]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='F1-Score',
    ascending=False
)

print("==========================================")
print("RESULTADOS FINALES")
print("==========================================")
print(results_df.to_string(index=False))
print()

RESULTADOS FINALES
                                    Modelo  Accuracy  Precision   Recall  F1-Score  Std Accuracy   Std F1
Random Forest (250 árboles / max_depth=30)  0.841967   0.989837 0.836056  0.906470      0.002769 0.001712
               Random Forest (300 árboles)  0.841534   0.989953 0.835479  0.906178      0.002815 0.001777
               Random Forest (200 árboles)  0.840900   0.989890 0.834833  0.905771      0.003299 0.002086
               Random Forest (150 árboles)  0.840466   0.989818 0.834418  0.905496      0.003255 0.002057
               Random Forest (100 árboles)  0.840033   0.989866 0.833899  0.905210      0.003427 0.002174
                Random Forest (50 árboles)  0.836863   0.989689 0.830553  0.903164      0.002529 0.001590
Random Forest (250 árboles / max_depth=20)  0.836028   0.990003 0.829365  0.902590      0.002946 0.001875
                          SVM (rbf kernel)  0.822407   0.993170 0.811703  0.893310      0.002613 0.001762
           Gradient Boostin